In [1]:
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np

In [2]:
# Reading source
f = open("Ahemdabad_source.txt","r",encoding="utf-8")
page_source = f.read()
f.close()

In [3]:
# parsing the soup using html parser
soup = BeautifulSoup(page_source,'html.parser')

In [4]:
# finds all the spc-img where we can get Title and Rera approvement
project_divs = soup.find_all('div', class_='spc-img')

In [5]:
# using img class extracted -> RERA and Project Name
projects_name_and_rera_registration= []
for div in project_divs:
    # get title from <img>
    img = div.find("img")
    # print(img.attrs)
    title = img["title"].strip() if img and "title" in img.attrs else None

    # get RERA status from <span>
    span = div.find("span", class_="spc-rr")
    # rera = span.get_text(strip=True) if span else "No"
    rera = "Yes" if (span and span.get_text(strip=True)) else "No"

    if title:
        projects_name_and_rera_registration.append({
            "title": title,
            "Rera registered": rera
        })

In [6]:
df_titleandrera = pd.DataFrame(projects_name_and_rera_registration)
df_titleandrera

,title,Rera registered
0,Ansh Aaradhya,Yes
1,Patrakar Heights,No
2,Poonam Pride,Yes
3,Shagun Saral 207,Yes
4,Express View City,No
...,...,...
985,Maruti Celosia Greenscapes 2,No
986,Magnanimous Pearl 2,No
987,Magnanimous Pearl,No
988,Magnanimous Pearl 3,No


In [7]:
Location_and_builder = [] # Location and builder are done

for i in range(len(project_divs)):
    temp = project_divs[i].find_next_sibling("div", class_="spc-info")
    A = temp.find(class_="spc-location").get_text().strip().split(",")
    Location_and_builder.append(A)

df_locandbuild = pd.DataFrame(Location_and_builder)
df_locandbuild.drop(2, axis=1,inplace=True) # Dropping none columne inplace
df_locandbuild.rename(columns={0: "Location",1:"Builder"},inplace=True)
df_locandbuild


,Location,Builder
0,Khoraj,Ahmedabad By Ansh Group
1,Naranpura,Ahmedabad\nBy Samyag Group
2,Shela,Ahmedabad\nBy B R Developers
3,Zundal,Ahmedabad\nBy aditya buildcon
4,Dholera,Ahmedabad\nBy Kavvan Infrra
...,...,...
985,Changodar,Ahmedabad\nBy Maruti Group
986,Dholera,Ahmedabad\nBy Magnanimous Infrastructure Pvt....
987,Dholera,Ahmedabad\nBy Magnanimous Infrastructure Pvt....
988,Dholera,Ahmedabad\nBy Magnanimous Infrastructure Pvt....


In [8]:
price_fetch = []
for i in range(len(project_divs)):
   div_1 = project_divs[i].find_next_sibling("div", class_="spc-info")
   
   if bool(div_1.find(class_="spc-price")) == True: # finds the class where price is, If true then appends the price
      price_fetch.append(div_1.find(class_="spc-price").get_text().strip())
   else:
      price_fetch.append("None") # else appends none

df_price = pd.DataFrame(price_fetch)
df_price.rename(columns={0:"Price"},inplace=True)
df_price

,Price
0,60 - 65 Lac
1,None
2,2.25 Cr.
3,None
4,None
...,...
985,None
986,5.40 Lac
987,4 - 17.98 Lac
988,6.30 - 32.29 Lac


In [9]:
def extract_project_info(info_div):
    info = {}
    
    # loop through li elements
    for li in info_div.select("ul.spc-feature li"):
        label = li.find("span", class_="spcf-lbl").get_text(strip=True)
        value = li.find("strong", class_="spcf-val sc").get_text(strip=True)
        info[label] = value
    
    # extract with defaults if missing
    return {
        "property_type": info.get("Configs"),
        "total_units": info.get("Total Units", None),
        "possession_status": info.get("Possession Status")
    }

all_projects = []

for div in project_divs:
    temp_div = div.find_next_sibling("div", class_="spc-info")
    if temp_div:  # check just in case some don't have info
        project_info = extract_project_info(temp_div)
        all_projects.append(project_info)

df_proptype_units_possesion = pd.DataFrame(all_projects)
df_proptype_units_possesion

,property_type,total_units,possession_status
0,2 BHK Apartment,91,Upcoming Projects
1,2 BHK Apartment,None,Upcoming Projects
2,4 BHK Apartment,172,Completed Projects
3,3 BHK Apartment,192,Upcoming Projects
4,Residential Plot,None,Upcoming Projects
...,...,...,...
985,Residential Plot,186,Completed Projects
986,Residential Plot,196,Completed Projects
987,Residential Plot,165,Completed Projects
988,Residential Plot,124,Completed Projects


In [10]:
df_realestatedata = pd.concat([df_titleandrera, df_proptype_units_possesion, df_price, df_locandbuild],axis=1)
# len(df_realestatedata[df_realestatedata["Builder"].str.contains("By", na=False)])

In [11]:
df_realestatedata["Builder"] = df_realestatedata["Builder"].mask(
    ~df_realestatedata["Builder"].str.contains("By", na=False), # '~' = Bit wise Not function , returns complement
    "Not known"
) # Replaces Builders not avaiblable by None

df_realestatedata["Builder"] = (
    df_realestatedata["Builder"]
    .str.split("By", n=1)
    .str[-1]        # take the part after "By"
    .str.strip()    # remove leading/trailing spaces/newlines
) # Splits on basis of By and only lets Builder remain  

In [12]:
df_realestatedata # Data frame done!

,title,Rera registered,property_type,total_units,possession_status,Price,Location,Builder
0,Ansh Aaradhya,Yes,2 BHK Apartment,91,Upcoming Projects,60 - 65 Lac,Khoraj,Ansh Group
1,Patrakar Heights,No,2 BHK Apartment,None,Upcoming Projects,None,Naranpura,Samyag Group
2,Poonam Pride,Yes,4 BHK Apartment,172,Completed Projects,2.25 Cr.,Shela,B R Developers
3,Shagun Saral 207,Yes,3 BHK Apartment,192,Upcoming Projects,None,Zundal,aditya buildcon
4,Express View City,No,Residential Plot,None,Upcoming Projects,None,Dholera,Kavvan Infrra
...,...,...,...,...,...,...,...,...
985,Maruti Celosia Greenscapes 2,No,Residential Plot,186,Completed Projects,None,Changodar,Maruti Group
986,Magnanimous Pearl 2,No,Residential Plot,196,Completed Projects,5.40 Lac,Dholera,Magnanimous Infrastructure Pvt. Ltd
987,Magnanimous Pearl,No,Residential Plot,165,Completed Projects,4 - 17.98 Lac,Dholera,Magnanimous Infrastructure Pvt. Ltd
988,Magnanimous Pearl 3,No,Residential Plot,124,Completed Projects,6.30 - 32.29 Lac,Dholera,Magnanimous Infrastructure Pvt. Ltd


In [13]:
# Block for finding out index of anything i want to test
id = df_realestatedata[df_realestatedata["Price"] == "1.10 - 1.30 Cr."].index[0]
df_realestatedata['Price'][id]

'1.10 - 1.30 Cr.'

In [14]:
def detect_unit(val):
    val = val.strip()
    if "Cr" in val:
        return "Crore"
    elif "Lac" in val or "lac" in val or "Lacs" in val:
        return "Lac"
    else:
        return None   # explicitly return None if nothing matched

def Price_unit(val1, val2):
    unit1 = detect_unit(val1)
    unit2 = detect_unit(val2)

    # Fallbacks
    if unit1 is None and unit2 is not None:
        unit1 = unit2
    if unit1 is None:
        unit1 = "Lac"
    if unit2 is None:
        unit2 = "Lac"

    return unit1, unit2

def convert_to_lakh(value_str, unit):
    """Convert a string like '1.10' with unit into Lakhs."""
    try:
        num = float(value_str.replace("Cr.", "").replace("Cr", "").replace("Lac", "").strip())
    except:
        return np.nan
    
    if unit == "Crore":
        return num * 100  # 1 Cr = 100 Lakhs
    elif unit == "Lac":
        return num        # already in Lakhs
    else:
        return num

def parse_price(value):
    if pd.isna(value) or value.strip() == "None":
        return (np.nan, np.nan, np.nan)

    val = value.replace(",", "").strip()

    if "-" in val:
        low, high = val.split("-")
        low_s, high_s = low.strip(), high.strip()
        
        # Detect units
        unit1, unit2 = Price_unit(low_s, high_s)

        # Convert separately
        low_val = convert_to_lakh(low_s, unit1)
        high_val = convert_to_lakh(high_s, unit2)

        if np.isnan(low_val) or np.isnan(high_val):
            return (np.nan, np.nan, np.nan)
        
        avg = (low_val + high_val) / 2
        return (low_val, high_val, avg)
    else:
        unit = detect_unit(val)
        num = convert_to_lakh(val, unit)
        return (num, num, num)
    
df_realestatedata[["Min Price","Max Price","Average Price"]] = (
    df_realestatedata['Price'].apply(parse_price).apply(pd.Series)
    )
